<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/09_Motor_de_Inferencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 09 - Motor de Inferência

## Implementação de Forward Chaining e Backward Chaining

Este notebook implementa um **motor de diagnóstico baseado em regras** para o projeto de um **Sistema SCADA aplicado a um Drone Agrícola de Pulverização**.

O motor recebe fatos derivados das variáveis de processo e utiliza regras lógicas para gerar diagnósticos, alarmes e ações recomendadas.

Serão implementados:

- **Forward Chaining** — parte dos fatos conhecidos e produz novas conclusões;
- **Backward Chaining** — parte de uma hipótese e verifica se ela pode ser comprovada;
- Conversão das medições dos sensores em fatos;
- Integração do motor com o diagnóstico do drone.


## 1. Definição das regras

Cada regra possui:

- um identificador;
- um conjunto de condições;
- uma conclusão;
- uma descrição.

A regra é ativada quando **todas as suas condições forem verdadeiras**.


In [25]:
regras = [
    {
        "nome": "R1",
        "condicoes": {"nivel_critico"},
        "conclusao": "falta_insumo",
        "descricao": "Nível crítico indica falta de insumo."
    },
    {
        "nome": "R2",
        "condicoes": {"bomba_ligada", "valvula_aberta", "vazao_baixa"},
        "conclusao": "falha_pulverizacao",
        "descricao": "Bomba e válvula ativas com vazão baixa indicam falha de pulverização."
    },
    {
        "nome": "R3",
        "condicoes": {"bomba_ligada", "vazao_baixa", "pressao_alta"},
        "conclusao": "possivel_obstrucao",
        "descricao": "Vazão baixa com pressão alta pode indicar obstrução."
    },
    {
        "nome": "R4",
        "condicoes": {"possivel_obstrucao"},
        "conclusao": "interromper_pulverizacao",
        "descricao": "Uma possível obstrução exige interrupção da pulverização."
    },
    {
        "nome": "R5",
        "condicoes": {"falta_insumo"},
        "conclusao": "interromper_pulverizacao",
        "descricao": "Falta de insumo exige interrupção da pulverização."
    },
    {
        "nome": "R6",
        "condicoes": {"bateria_critica"},
        "conclusao": "interromper_missao",
        "descricao": "Bateria crítica exige interrupção da missão."
    },
    {
        "nome": "R7",
        "condicoes": {"vento_alto"},
        "conclusao": "suspender_pulverizacao",
        "descricao": "Vento alto torna a pulverização inadequada."
    },
    {
        "nome": "R8",
        "condicoes": {"bomba_ligada", "vazao_baixa", "pressao_baixa"},
        "conclusao": "possivel_vazamento_ou_falha_bomba",
        "descricao": "Vazão e pressão baixas com a bomba ligada podem indicar vazamento ou falha da bomba."
    }
]

for regra in regras:
    print(
        f'{regra["nome"]}: '
        f'SE {" E ".join(sorted(regra["condicoes"]))} '
        f'ENTÃO {regra["conclusao"]}'
    )


R1: SE nivel_critico ENTÃO falta_insumo
R2: SE bomba_ligada E valvula_aberta E vazao_baixa ENTÃO falha_pulverizacao
R3: SE bomba_ligada E pressao_alta E vazao_baixa ENTÃO possivel_obstrucao
R4: SE possivel_obstrucao ENTÃO interromper_pulverizacao
R5: SE falta_insumo ENTÃO interromper_pulverizacao
R6: SE bateria_critica ENTÃO interromper_missao
R7: SE vento_alto ENTÃO suspender_pulverizacao
R8: SE bomba_ligada E pressao_baixa E vazao_baixa ENTÃO possivel_vazamento_ou_falha_bomba


## 2. Algoritmo de Forward Chaining

O algoritmo começa com os fatos conhecidos e percorre repetidamente as regras.

Quando todas as condições de uma regra estão presentes, sua conclusão é adicionada à base de fatos. O processo termina quando nenhuma nova informação pode ser inferida.


In [26]:
def forward_chaining(fatos_iniciais, regras):
    fatos = set(fatos_iniciais)
    trilha = []
    houve_alteracao = True

    while houve_alteracao:
        houve_alteracao = False

        for regra in regras:
            condicoes = regra["condicoes"]
            conclusao = regra["conclusao"]

            if condicoes.issubset(fatos) and conclusao not in fatos:
                fatos.add(conclusao)
                trilha.append({
                    "regra": regra["nome"],
                    "condicoes": sorted(condicoes),
                    "conclusao": conclusao
                })
                houve_alteracao = True

    return fatos, trilha


## 3. Teste simples do Forward Chaining

Neste primeiro teste, considera-se:

- bomba ligada;
- vazão baixa;
- pressão alta.

O resultado esperado é a identificação de uma **possível obstrução** e, posteriormente, a conclusão de que a pulverização deve ser interrompida.


In [27]:
fatos_teste = {
    "bomba_ligada",
    "vazao_baixa",
    "pressao_alta"
}

fatos_resultantes, trilha = forward_chaining(fatos_teste, regras)

print("Fatos iniciais:")
print(sorted(fatos_teste))

print("\nFatos após a inferência:")
print(sorted(fatos_resultantes))

print("\nRegras ativadas:")
for passo in trilha:
    print(
        f'{passo["regra"]}: '
        f'{", ".join(passo["condicoes"])} -> {passo["conclusao"]}'
    )


Fatos iniciais:
['bomba_ligada', 'pressao_alta', 'vazao_baixa']

Fatos após a inferência:
['bomba_ligada', 'interromper_pulverizacao', 'possivel_obstrucao', 'pressao_alta', 'vazao_baixa']

Regras ativadas:
R3: bomba_ligada, pressao_alta, vazao_baixa -> possivel_obstrucao
R4: possivel_obstrucao -> interromper_pulverizacao


## 4. Algoritmo de Backward Chaining

O Backward Chaining recebe um objetivo, por exemplo:

`possivel_obstrucao`

O algoritmo procura uma regra cuja conclusão seja esse objetivo e verifica se todas as condições necessárias podem ser comprovadas.


In [28]:
def backward_chaining(objetivo, fatos, regras, visitados=None, profundidade=0):
    if visitados is None:
        visitados = set()

    trilha = []
    indentacao = "  " * profundidade

    if objetivo in fatos:
        trilha.append(f"{indentacao}Fato conhecido: {objetivo}")
        return True, trilha

    if objetivo in visitados:
        trilha.append(f"{indentacao}Objetivo já visitado: {objetivo}")
        return False, trilha

    visitados = visitados | {objetivo}

    regras_objetivo = [
        regra for regra in regras
        if regra["conclusao"] == objetivo
    ]

    if not regras_objetivo:
        trilha.append(f"{indentacao}Nenhuma regra conclui: {objetivo}")
        return False, trilha

    for regra in regras_objetivo:
        trilha.append(
            f'{indentacao}Testando {regra["nome"]} para concluir {objetivo}'
        )

        todas_satisfeitas = True

        for condicao in regra["condicoes"]:
            resultado, subtrilha = backward_chaining(
                condicao,
                fatos,
                regras,
                visitados,
                profundidade + 1
            )
            trilha.extend(subtrilha)

            if not resultado:
                todas_satisfeitas = False
                break

        if todas_satisfeitas:
            trilha.append(
                f'{indentacao}{regra["nome"]} satisfeita -> {objetivo}'
            )
            return True, trilha

    trilha.append(f"{indentacao}Não foi possível comprovar: {objetivo}")
    return False, trilha


In [29]:
objetivo = "possivel_obstrucao"

resultado, trilha_backward = backward_chaining(
    objetivo,
    fatos_teste,
    regras
)

print(f"Objetivo: {objetivo}")
print(f"Comprovado? {resultado}")

print("\nTrilha de inferência:")
for linha in trilha_backward:
    print(linha)


Objetivo: possivel_obstrucao
Comprovado? True

Trilha de inferência:
Testando R3 para concluir possivel_obstrucao
  Fato conhecido: vazao_baixa
  Fato conhecido: bomba_ligada
  Fato conhecido: pressao_alta
R3 satisfeita -> possivel_obstrucao


## 5. Conversão das medições em fatos

Os sensores fornecem valores numéricos, porém o motor de inferência trabalha com fatos lógicos.

A função abaixo converte as medições em fatos a partir de limites pré-definidos.

Os limites utilizados são apenas valores de exemplo e podem ser ajustados posteriormente.


In [30]:
def medicoes_para_fatos(medicoes):
    fatos = set()

    nivel = medicoes["nivel_reservatorio"]
    vazao = medicoes["vazao"]
    pressao = medicoes["pressao"]
    bateria = medicoes["bateria"]
    vento = medicoes["vento"]

    if medicoes.get("bomba_ligada", False):
        fatos.add("bomba_ligada")

    if medicoes.get("valvula_aberta", False):
        fatos.add("valvula_aberta")

    # Nível do reservatório
    if nivel < 20:
        fatos.add("nivel_baixo")

    if nivel < 5:
        fatos.add("nivel_critico")

    # Vazão
    if vazao < 0.5 and medicoes.get("bomba_ligada", False):
        fatos.add("vazao_baixa")
    else:
        fatos.add("vazao_normal")

    # Pressão
    if pressao > 5:
        fatos.add("pressao_alta")
    elif pressao < 1 and medicoes.get("bomba_ligada", False):
        fatos.add("pressao_baixa")
    else:
        fatos.add("pressao_normal")

    # Bateria
    if bateria < 20:
        fatos.add("bateria_baixa")

    if bateria < 10:
        fatos.add("bateria_critica")

    # Vento
    if vento > 8:
        fatos.add("vento_alto")
    else:
        fatos.add("vento_adequado")

    return fatos


## 6. Cenário de diagnóstico integrado

Neste cenário, o drone apresenta:

- reservatório praticamente vazio;
- bomba ligada;
- válvula aberta;
- vazão baixa;
- pressão elevada;
- bateria em condição normal;
- vento dentro do limite.

O motor deve detectar tanto a falta de insumo quanto uma possível obstrução.


In [31]:
medicoes = {
    "nivel_reservatorio": 4.0,  # %
    "vazao": 0.2,              # L/min
    "pressao": 5.2,            # bar
    "bateria": 55.0,           # %
    "vento": 4.0,              # m/s
    "bomba_ligada": True,
    "valvula_aberta": True
}

fatos_sensores = medicoes_para_fatos(medicoes)

print("Fatos gerados a partir dos sensores:")
for fato in sorted(fatos_sensores):
    print("-", fato)


Fatos gerados a partir dos sensores:
- bomba_ligada
- nivel_baixo
- nivel_critico
- pressao_alta
- valvula_aberta
- vazao_baixa
- vento_adequado


In [32]:
fatos_diagnosticados, trilha_diagnostico = forward_chaining(
    fatos_sensores,
    regras
)

print("Diagnóstico completo:")
for fato in sorted(fatos_diagnosticados):
    print("-", fato)

print("\nRegras ativadas:")
for passo in trilha_diagnostico:
    print(
        f'{passo["regra"]}: '
        f'{" + ".join(passo["condicoes"])} '
        f'-> {passo["conclusao"]}'
    )


Diagnóstico completo:
- bomba_ligada
- falha_pulverizacao
- falta_insumo
- interromper_pulverizacao
- nivel_baixo
- nivel_critico
- possivel_obstrucao
- pressao_alta
- valvula_aberta
- vazao_baixa
- vento_adequado

Regras ativadas:
R1: nivel_critico -> falta_insumo
R2: bomba_ligada + valvula_aberta + vazao_baixa -> falha_pulverizacao
R3: bomba_ligada + pressao_alta + vazao_baixa -> possivel_obstrucao
R4: possivel_obstrucao -> interromper_pulverizacao


## 7. Apresentação dos diagnósticos para o SCADA

As conclusões do motor podem ser traduzidas para mensagens mais amigáveis para exibição na HMI.


In [33]:
mensagens_diagnostico = {
    "falta_insumo": "Nível crítico de calda. Realizar abastecimento.",
    "falha_pulverizacao": "Falha no sistema de pulverização.",
    "possivel_obstrucao": "Possível obstrução nos bicos ou na tubulação.",
    "possivel_vazamento_ou_falha_bomba": "Possível vazamento ou falha da bomba.",
    "interromper_pulverizacao": "Interromper a pulverização.",
    "interromper_missao": "Interromper a missão e retornar à base.",
    "suspender_pulverizacao": "Suspender a pulverização devido às condições ambientais."
}

diagnosticos = [
    mensagens_diagnostico[fato]
    for fato in fatos_diagnosticados
    if fato in mensagens_diagnostico
]

print("Mensagens para a HMI:")
for mensagem in diagnosticos:
    print("-", mensagem)


Mensagens para a HMI:
- Possível obstrução nos bicos ou na tubulação.
- Falha no sistema de pulverização.
- Nível crítico de calda. Realizar abastecimento.
- Interromper a pulverização.


## 8. Verificação de uma hipótese específica

O Backward Chaining pode ser utilizado quando o sistema deseja confirmar um diagnóstico específico.

A seguir, verifica-se se existe evidência suficiente para concluir que a pulverização deve ser interrompida.


In [34]:
objetivo = "interromper_pulverizacao"

resultado, trilha = backward_chaining(
    objetivo,
    fatos_sensores,
    regras
)

print(f"Hipótese: {objetivo}")
print(f"Resultado: {resultado}")

print("\nJustificativa:")
for linha in trilha:
    print(linha)


Hipótese: interromper_pulverizacao
Resultado: True

Justificativa:
Testando R4 para concluir interromper_pulverizacao
  Testando R3 para concluir possivel_obstrucao
    Fato conhecido: vazao_baixa
    Fato conhecido: bomba_ligada
    Fato conhecido: pressao_alta
  R3 satisfeita -> possivel_obstrucao
R4 satisfeita -> interromper_pulverizacao


## 9. Função integrada do motor de diagnóstico

A função abaixo reúne as principais etapas:

1. Recebe as medições;
2. Converte os valores em fatos;
3. Executa o Forward Chaining;
4. Retorna fatos, diagnósticos e regras ativadas.

Essa função representa uma possível interface entre o sistema SCADA e o motor de inferência.


In [35]:
def motor_diagnostico(medicoes, regras):
    fatos_iniciais = medicoes_para_fatos(medicoes)

    fatos_finais, trilha = forward_chaining(
        fatos_iniciais,
        regras
    )

    diagnosticos = [
        {
            "codigo": fato,
            "mensagem": mensagens_diagnostico[fato]
        }
        for fato in fatos_finais
        if fato in mensagens_diagnostico
    ]

    return {
        "medicoes": medicoes,
        "fatos_iniciais": sorted(fatos_iniciais),
        "fatos_finais": sorted(fatos_finais),
        "diagnosticos": diagnosticos,
        "regras_ativadas": trilha
    }


In [36]:
resultado = motor_diagnostico(medicoes, regras)

print("=== MOTOR DE DIAGNÓSTICO ===")

print("\nFatos iniciais:")
for fato in resultado["fatos_iniciais"]:
    print("-", fato)

print("\nDiagnósticos:")
for diagnostico in resultado["diagnosticos"]:
    print(
        f'- [{diagnostico["codigo"]}] '
        f'{diagnostico["mensagem"]}'
    )

print("\nRegras ativadas:")
for regra in resultado["regras_ativadas"]:
    print(
        f'- {regra["regra"]}: '
        f'{", ".join(regra["condicoes"])} '
        f'-> {regra["conclusao"]}'
    )


=== MOTOR DE DIAGNÓSTICO ===

Fatos iniciais:
- bomba_ligada
- nivel_baixo
- nivel_critico
- pressao_alta
- valvula_aberta
- vazao_baixa
- vento_adequado

Diagnósticos:
- [possivel_obstrucao] Possível obstrução nos bicos ou na tubulação.
- [falha_pulverizacao] Falha no sistema de pulverização.
- [falta_insumo] Nível crítico de calda. Realizar abastecimento.
- [interromper_pulverizacao] Interromper a pulverização.

Regras ativadas:
- R1: nivel_critico -> falta_insumo
- R2: bomba_ligada, valvula_aberta, vazao_baixa -> falha_pulverizacao
- R3: bomba_ligada, pressao_alta, vazao_baixa -> possivel_obstrucao
- R4: possivel_obstrucao -> interromper_pulverizacao


## Conclusão

O motor implementado demonstra a utilização de **Forward Chaining** e **Backward Chaining** para diagnóstico de condições do drone agrícola.

O Forward Chaining é especialmente apropriado para a integração contínua com o SCADA, pois parte diretamente dos fatos gerados pelos sensores e identifica automaticamente diagnósticos e ações.

O Backward Chaining permite verificar hipóteses específicas, sendo útil para justificar ao operador por que determinado diagnóstico foi ou não confirmado.

A estrutura apresentada pode ser expandida com novas regras e novos fatos à medida que outras variáveis da planta forem incorporadas ao projeto.
